# 0901 2일차

## 0. 파이썬의 import 방식

파이썬은 자바처럼 패키지를 통째로 가져오지 않고, `from A import B`처럼 필요한 것만 가져오는 방식을 기본으로 씀

```java
import java.util.*;
```

```python
import tensorflow as tf
from tensorflow.keras.models import Sequential
```

![자바와 파이썬의 import 방식 비교](assets/import-java-vs-python.svg)

### 0-1. 필요한 것만 가져오는 이유

**자바의 경우**
- 컴파일 시점에 클래스 이름이 어떤 파일(클래스)에 대응하는지 컴파일러가 미리 다 알고 있음
- 그래서 와일드카드(`*`)로 한꺼번에 가져와도 실행 비용이나 이름 충돌 문제가 거의 없음

**파이썬의 경우**
1. 모듈 동작 원리 : 파이썬의 `import`는 모듈 파일을 실제로 한 번 실행하는 행위임
   - `import tensorflow`를 하면 tensorflow의 `__init__.py`가 처음부터 끝까지 실행되며 변수·함수·클래스가 메모리에 올라감
   - tensorflow처럼 서브모듈이 수천 개인 라이브러리를 `from tensorflow import *`로 다 가져오면 필요 없는 것까지 메모리에 올라가 느리고 무거워짐
2. 이름 충돌 : 모듈 안의 이름들이 평평한(flat) 하나의 이름 공간으로 들어옴
   - 평평하다는 것은 패키지 → 클래스 같은 계층 없이 모든 이름이 같은 레벨에 놓인다는 뜻
   - 두 라이브러리에 같은 이름(`max`, `min`, `array` 등)이 있으면 나중에 import한 것이 앞의 것을 에러 없이 덮어씀
   - 자바는 클래스명이 겹치는 일이 드물고, 겹쳐도 컴파일러가 잡아냄
3. 명시성 : `from tensorflow.keras.models import Sequential`이 `from tensorflow import *`보다 어디서 온 이름인지 명확함

```python
from tensorflow import *
from numpy import *        # 같은 이름이 있으면 앞의 것을 덮어씀
```

| | 자바 `*` | 파이썬 `import` |
|---|---|---|
| 하는 일 | 컴파일러에 이름표 등록 | 모듈 코드를 실제로 실행 |
| 이름 충돌 | 위험 낮음 | 평평한 네임스페이스라 위험 높음 |
| 실행 비용 | 없음 | 있음 |

### 0-2. 함수와 클래스

파이썬에서 이름 뒤에 `()`가 붙으면 호출한다는 뜻이고, 함수일 수도 클래스일 수도 있음

**구분 방법 : 첫 글자**
1. `소문자()` : 함수 → 실행하고 결과를 돌려줌 (`print()`, `len()`, `np.array()`)
2. `대문자()` : 클래스 → 객체(인스턴스)를 만들어서 돌려줌 (`Sequential()`, `Dense()`)

```python
model = Sequential()     # Sequential 클래스의 객체를 만들어 model에 담음
```

**자바와 비교**
- 자바는 `new` 키워드로 객체를 생성함 : `Sequential model = new Sequential();`
- 파이썬은 `new` 없이 클래스 이름에 `()`를 붙이면 객체가 만들어짐

**이름 규칙 (PEP8)**

| 대상 | 표기법 | 예시 |
|---|---|---|
| 함수 / 변수 | snake_case (소문자 + 언더바) | `train_test_split`, `input_dim` |
| 클래스 | PascalCase (단어마다 대문자) | `Sequential`, `EarlyStopping` |
| 상수 | 전부 대문자 | `MAX_SIZE` |

- 문법으로 강제되는 것이 아니라 관례임 (소문자로 클래스를 만들어도 에러는 안 남)
- 파이썬 표준 라이브러리, numpy, 케라스가 모두 이 규칙을 지켜서 대문자로 시작하면 클래스라고 봐도 거의 맞음

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential

print(type(np.array))     # <class 'builtin_function_or_method'> - 소문자 → 함수
print(type(Sequential))   # <class 'type'>                       - 대문자 → 클래스

model = Sequential()
print(type(model))        # <class '...Sequential'> - 클래스로 만들어진 객체(인스턴스)

## 1. 스칼라 / 벡터 / 행렬 / 텐서

데이터를 몇 차원으로 모았는지에 따라 부르는 이름

**종류**
1. 스칼라 (Scalar) : 숫자 하나 → `1`, `2`, `3` 각각
2. 벡터 (Vector) : 스칼라를 일렬로 나열한 모음 (1차원) → `[1, 2, 3]`
3. 행렬 (Matrix) : 벡터를 모은 것 (2차원) → `[[1,2,3],[4,5,6],[7,8,9]]`
4. 텐서 (Tensor) : 행렬을 여러 개 쌓아 더 높은 차원으로 확장한 것 (N차원)

TensorFlow의 모든 연산은 다차원 행렬(텐서) 연산임

### 1-1. 텐서의 차원 (rank)

스칼라·벡터·행렬도 모두 텐서의 특수한 경우이고, 차원(rank)만 다름

| 차원(rank) | 이름 |
|---|---|
| 0차원 | 스칼라 |
| 1차원 | 벡터 |
| 2차원 | 행렬 |
| 3차원 이상 | 텐서 (3D 텐서, 4D 텐서...로 차원 수만 구분) |

- 행렬(2D)을 모으면 3D 텐서, 3D 텐서를 모으면 4D 텐서가 됨
- 예: 컬러 이미지 1장 = 3D 텐서 (가로×세로×RGB), 여러 장을 묶은 배치 = 4D 텐서 (장수×가로×세로×RGB)

In [ ]:
import tensorflow as tf

a = tf.constant(5)                 # 스칼라
b = tf.constant([1, 2, 3])         # 벡터
c = tf.constant([[1, 2], [3, 4]])  # 행렬

print(type(a), type(b), type(c))   # 셋 다 <class '...EagerTensor'> - 동일한 타입
print(a.shape, b.shape, c.shape)   # (), (3,), (2, 2) - shape(차원)만 다름

- TensorFlow는 스칼라·벡터·행렬을 모두 같은 `Tensor` 클래스로 다루고, 차이는 `.shape`(차원 구조)뿐임

## 2. shape

데이터의 각 차원별 크기를 괄호 안에 순서대로 적은 것

**shape가 필요한 이유**
- 스칼라·벡터·행렬·텐서를 이름으로 구분한 것을 숫자로 정확하게 표현함
- 케라스는 이 shape를 보고 레이어의 입출력을 맞춤

| 구분 | 값 | shape |
|---|---|---|
| 벡터 | `[1, 2, 3]` | `(3,)` |
| 행렬 | `[[1, 2, 3], [4, 5, 6]]` | `(2, 3)` |
| 텐서 | `[[[1,2,3],[4,5,6]], [[3,7,11],[5,4,3]]]` | `(2, 2, 3)` |

- `(3,)` : 원소 3개짜리 1차원
- `(2, 3)` : 2행 3열 2차원
- shape는 왼쪽부터 바깥 대괄호 → 안쪽 대괄호 순서로 적음

### 2-1. shape 읽는 법

**순서**
1. 가장 안쪽 대괄호(벡터)의 원소가 몇 개인지 셈
2. 그 벡터를 묶은 단위(행렬) 안에 벡터가 몇 개인지 셈
3. 그 행렬을 묶은 단위(텐서) 안에 행렬이 몇 개인지 셈
4. 센 결과를 바깥 → 안쪽 순서로 거꾸로 나열함

**예시** `[[[1,2,3],[4,5,6]], [[3,7,11],[5,4,3]]]`
- 벡터 원소 개수 : 3
- 행렬 안 벡터 개수 : 2
- 텐서 안 행렬 개수 : 2
- shape = `(2, 2, 3)`

## 3. reshape

데이터가 저장된 순서는 그대로 두고, 몇 개씩 끊어서 볼지(shape)만 바꾸는 것

- 값이 바뀌거나 복사되는 것이 아니라 대괄호를 다시 치는 것에 가까움

| 값 | shape | 읽는 법 |
|---|---|---|
| `[1, 2, 3]` | `(3,)` | 원소 3개짜리 벡터 (1차원) |
| `[[1], [2], [3]]` | `(3, 1)` | 3행 1열 행렬 (2차원) |
| `[[1, 2, 3]]` | `(1, 3)` | 1행 3열 행렬 (2차원) |

- 셋 다 안에 든 숫자는 `1, 2, 3`으로 같고, 묶는 방법만 다름

In [ ]:
import numpy as np

a = np.array([1, 2, 3])

print(a.shape)                # (3,)   - 벡터
print(a.reshape(3, 1).shape)  # (3, 1) - 3행 1열
print(a.reshape(1, 3).shape)  # (1, 3) - 1행 3열

print(a.reshape(3, 1))
# [[1]
#  [2]
#  [3]]

### 3-1. reshape 규칙

1. 전체 원소 개수(shape를 모두 곱한 값)가 같아야 함
   - `(3,)` → 3개, `(3, 1)` → 3개, `(1, 3)` → 3개 → 서로 변환 가능
   - `(3,)` → `(2, 2)`는 불가능 (3개를 4개로 만들 수 없음) → `ValueError`
2. `-1`은 "나머지 크기를 알아서 계산하라"는 뜻
   - 데이터 개수를 일일이 세지 않아도 되어 실전에서 자주 씀

```python
x.reshape(-1, 1)   # 열은 1개로 고정, 행 개수는 알아서 계산
x.reshape(1, -1)   # 행은 1개로 고정, 열 개수는 알아서 계산
```

### 3-2. cf) reshape와 전치(.T)의 차이

`.T`(전치)도 shape를 바꾸지만 reshape와 다른 연산임

| | 하는 일 | `[[1,2,3],[4,5,6]]` (2,3) → (3,2) |
|---|---|---|
| `reshape(3, 2)` | 값을 저장된 순서 그대로 다시 끊어 읽음 | `[[1,2],[3,4],[5,6]]` |
| `.T` (전치) | 행과 열을 바꿈 (축 교환) | `[[1,4],[2,5],[3,6]]` |

- shape는 `(3, 2)`로 같아도 값이 놓이는 자리가 다름
- `(특성, 샘플)`로 적은 데이터를 `(샘플, 특성)`으로 돌릴 때는 `.T`를 써야 함 (`keras/keras08_mlp1_2.py`)

### 3-3. 케라스 입력 규칙 - Dense의 2차원 입력

`Dense` 레이어는 입력을 항상 `(샘플 개수, 특성 개수)` 2차원으로 받음

**1차원 입력이 모호한 이유**
- `(3,)`짜리 `[1, 2, 3]`은 샘플 3개·특성 1개 `(3, 1)`인지, 샘플 1개·특성 3개 `(1, 3)`인지 데이터만 봐서는 알 수 없음

**케라스의 처리 규칙**
1. 1차원을 넣어도 에러가 나지 않음
2. 케라스가 마지막에 크기 1인 축을 붙여 `(3, 1)`로 해석함 (= `reshape(-1, 1)`을 대신 해줌)
3. 전치(`.T`)는 대신 해주지 않음 (§3-2)

`Dense(3, input_dim=1)` 모델에 직접 넣어본 결과

| 넣은 `x` | 케라스 해석 | 결과 |
|---|---|---|
| `(3,)` | 샘플 3개 × 특성 1개 | 출력 `(3, 1)` |
| `(3, 1)` | 샘플 3개 × 특성 1개 | 출력 `(3, 1)` — `(3,)`과 값까지 같음 |
| `(1, 3)` | 샘플 1개 × 특성 3개 | `ValueError` (`input_dim=1`과 불일치) |

- "샘플 1개 × 특성 3개"를 의도했더라도 케라스는 알 방법이 없어, 틀린 해석이어도 에러가 안 남
- 그래서 reshape로 어느 쪽인지 명시함 (`reshape(-1, 1)`은 의도 표시용)

**행과 열의 의미**
1. 행 = 데이터(샘플) 개수
2. 열 = 특성 개수 → `input_dim`에 들어가는 값

| shape | 의미 | `input_dim` |
|---|---|---|
| `(5, 1)` | 데이터 5개, 각각 특성 1개 | `1` |
| `(5, 2)` | 데이터 5개, 각각 특성 2개 | `2` |

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4, 5])
x = x.reshape(-1, 1)   # (5, 1) - 데이터 5개, 특성 1개

print(x.shape)         # (5, 1) → input_dim=1

# 특성이 2개인 경우
x2 = np.array([[1, 6], [2, 7], [3, 8], [4, 9], [5, 10]])
print(x2.shape)        # (5, 2) → input_dim=2

## 4. train / test 분리

전체 데이터를 훈련용과 평가용으로 나누어, 훈련에 쓰지 않은 데이터로 모델을 평가하는 것

**분리가 필요한 이유**
- `fit(x, y)`로 학습하고 `evaluate(x, y)`로 평가하면 훈련에 쓴 데이터로 평가한 것임
- loss가 작아도 처음 보는 데이터를 잘 맞히는지는 알 수 없음
- 과적합이 일어나도 탐지할 수 없음 (§4-1)

### 4-1. 과적합의 원인

과적합(overfitting) : 모델이 규칙을 배우지 않고 훈련 데이터를 외워버린 상태

1. 모델의 표현력 > 데이터의 양
   - `keras08_mlp4` 예 : 파라미터 204개(20+66+49+48+21) vs 학습에 쓴 y값 30개(10행 × 3열)
   - 미지수 204개에 식이 30개라 훈련 데이터를 완벽히 통과하는 해가 무수히 많음
   - 모델은 그중 하나를 고를 뿐, 그것이 진짜 규칙이라는 보장이 없음
   - 남는 파라미터로 규칙 대신 개별 데이터를 외움
2. 학습 목표가 훈련 데이터 loss 최소화뿐임
   - 일반화하라는 목표가 없어서, 외우는 쪽이 loss를 더 빨리 줄이면 모델은 그쪽을 택함
3. 노이즈까지 신호로 학습함
   - 규칙은 초반 epoch에 대부분 배우고, 이후 epoch는 노이즈를 맞추는 데 씀

### 4-2. train / test 분리 규칙

1. 전체 데이터의 약 70%를 훈련에, 나머지 약 30%를 평가에 씀
2. test는 학습에 한 번도 관여하면 안 됨
   - 한 번이라도 `fit`에 들어가면 평가의 의미가 사라짐

| 데이터 | 쓰는 곳 | 모델 입장 |
|---|---|---|
| `x_train`, `y_train` | `model.fit()` | 보고 배우는 데이터 |
| `x_test`, `y_test` | `model.evaluate()` | 처음 보는 데이터 |

**결과 진단**

| train loss | test loss | 진단 |
|---|---|---|
| 작다 | 작다 | 잘 학습됨 |
| 작다 | **크다** | **과적합** (외웠음) |
| 크다 | 크다 | 과소적합 (못 배웠음) |

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# 1. 데이터 - 10개 중 7개는 훈련용, 3개는 평가용으로 미리 잘라둠
x_train = np.array([1, 2, 3, 4, 5, 6, 7])
y_train = np.array([1, 2, 3, 4, 5, 6, 7])

x_test = np.array([8, 9, 10])
y_test = np.array([8, 9, 10])

# 2. 모델
model = Sequential()
model.add(Dense(1, input_dim=1))

# 3. 컴파일, 훈련 - train만 사용
model.compile(loss="mse", optimizer="adam")
model.fit(x_train, y_train, epochs=100, batch_size=4)

# 4. 평가 - test만 사용 (모델이 처음 보는 데이터)
loss = model.evaluate(x_test, y_test)
print("loss : ", loss)

### 4-3. 주의) 순서대로 자르면 외삽이 됨

**문제**
- 위 코드는 x가 1~10으로 정렬된 상태에서 앞 7개를 train, 뒤 3개를 test로 잘랐음
- test(8, 9, 10)는 훈련에서 본 적 없는 구간이라 평가가 아니라 외삽(extrapolation)이 됨
- 결과가 나빠도 과적합 때문인지 범위 밖이라서인지 구분할 수 없음

**해결**
- 자르기 전에 섞어야(shuffle) train과 test가 같은 분포를 가짐
- 실전에서는 직접 자르지 않고 `train_test_split`을 씀 (3일차 §2)